# 16 — Competition-Level Feature Research

Bu notebook 01–15 arasındaki çalışmaların devamıdır. Amaç yeni model aileleri denemek değil,
**Notebook 11 (`11_submission_optimization.ipynb`) temporal OOF sistemini sabit tutarak**
feature representation sınırlarını ölçmektir.

Kurallar: seed=42, temporal OOF, known/masked ayrımı, fold-train dışında target istatistiği yok,
her aday tek başına baseline'a eklenir, ara sonuçlar checkpoint edilir. Veri şemasında
`SOIL_MOISTURE_tp1` bulunmadığı için tp1 içeren öneriler geleceği kullanmadan hesaplanamaz ve
bilimsel olarak reddedilir.

In [1]:
from pathlib import Path
from time import perf_counter
import json, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.metrics import root_mean_squared_error
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer
from sklearn.inspection import permutation_importance
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED)
pd.set_option("display.max_columns",120)
sns.set_theme(style="whitegrid")

def find_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/"data"/"raw"/"Train.csv").exists(): return p
    raise FileNotFoundError("Repository root bulunamadı")

ROOT=find_root(); DATA=ROOT/"data"/"raw"; SUB=ROOT/"submissions"
ART=ROOT/"artifacts"/"notebook16"; ART.mkdir(parents=True,exist_ok=True); SUB.mkdir(exist_ok=True)
T0=perf_counter()
def stage(name,t):
    sec=perf_counter()-t
    print(f"[STAGE] {name}: {sec:.1f}s | total={perf_counter()-T0:.1f}s")
    return perf_counter()
print("ROOT",ROOT)

ROOT C:\global-water-storage-forecasting


## 1. Repository audit, data and exact Notebook 11 baseline

In [2]:
t=perf_counter()
expected=[f"{i:02d}_" for i in range(1,16)]
audit=[]
for prefix in expected:
    paths=sorted((ROOT/"notebooks").glob(prefix+"*.ipynb"))
    for p in paths:
        raw=json.loads(p.read_text(encoding="utf-8"))
        audit.append({"notebook":p.name,"cells":len(raw["cells"]),
                      "executed_code_cells":sum(bool(c.get("outputs")) for c in raw["cells"] if c["cell_type"]=="code")})
audit_df=pd.DataFrame(audit)
display(audit_df)
assert all(any(x["notebook"].startswith(p) for x in audit) for p in expected)

train=pd.read_csv(DATA/"Train.csv",parse_dates=["time"])
test=pd.read_csv(DATA/"Test.csv",parse_dates=["time"])
sample=pd.read_csv(DATA/"SampleSubmission.csv")
test["_original_order"]=np.arange(len(test))
assert train.sample_id.is_unique and test.ID.is_unique and sample.ID.is_unique and test.ID.equals(sample.ID)
print("shapes",train.shape,test.shape,sample.shape,"masked test share",test.TWS_t.isna().mean())
t=stage("audit + load",t)

,notebook,cells,executed_code_cells
0,01_eda.ipynb,20,17
1,02_baseline_model.ipynb,22,7
2,03_lightgbm.ipynb,30,19
3,04_xgboost.ipynb,18,8
4,05_catboost.ipynb,22,9
5,06_ensemble.ipynb,19,5
6,07_feature_engineering.ipynb,22,8
7,08_time_series_validation.ipynb,8,2
8,09_advanced_model_selection.ipynb,34,15
9,10_model_explainability.ipynb,29,16


shapes (2154021, 13) (280961, 14) (280961, 2) masked test share 0.6652631504016572
[STAGE] audit + load: 4.6s | total=4.6s


In [3]:
SPEI=["SPEI_01_t","SPEI_03_t","SPEI_06_t","SPEI_12_t"]
def static_features(df):
    o=df.copy(); dt=pd.to_datetime(o.time)
    o["month"]=dt.dt.month.astype("int8"); o["quarter"]=dt.dt.quarter.astype("int8")
    o["year"]=dt.dt.year.astype("int16")
    o["elapsed_month"]=((dt.dt.year-train.time.dt.year.min())*12+dt.dt.month-1).astype("int16")
    o["month_sin_calc"]=np.sin(2*np.pi*o.month/12); o["month_cos_calc"]=np.cos(2*np.pi*o.month/12)
    o["season_sin"]=np.sin(2*np.pi*o.quarter/4); o["season_cos"]=np.cos(2*np.pi*o.quarter/4)
    s=o[SPEI]; o["spei_mean"]=s.mean(axis=1); o["spei_min"]=s.min(axis=1)
    o["spei_max"]=s.max(axis=1); o["spei_std"]=s.std(axis=1)
    o["spei_01_minus_12"]=o.SPEI_01_t-o.SPEI_12_t
    o["spei_03_minus_12"]=o.SPEI_03_t-o.SPEI_12_t
    o["spei_short_long_ratio"]=o.SPEI_01_t/(o.SPEI_12_t.abs()+.1)
    o["lat_band_5"]=np.floor((o.lat+90)/5); o["lon_band_5"]=np.floor((o.lon+180)/5)
    o["spatial_bin_10"]=o.lat_band_5*100+o.lon_band_5
    o["lat_lon_interaction"]=o.lat*o.lon/1000
    o["lat_sin"]=np.sin(np.radians(o.lat)); o["lat_cos"]=np.cos(np.radians(o.lat))
    o["lon_sin"]=np.sin(np.radians(o.lon)); o["lon_cos"]=np.cos(np.radians(o.lon))
    return o

train_f=static_features(train); test_f=static_features(test)
A=["lat","lon","TWS_t",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year"]
B=["quarter","elapsed_month","month_sin_calc","month_cos_calc","season_sin","season_cos"]
C=["spei_mean","spei_min","spei_max","spei_std","spei_01_minus_12","spei_03_minus_12","spei_short_long_ratio"]
D=["lat_band_5","lon_band_5","spatial_bin_10","lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
BASE_FEATURES=list(dict.fromkeys(A+B+C+D)); BASE_MASKED=[x for x in BASE_FEATURES if x!="TWS_t"]
PARAMS=dict(n_estimators=160,learning_rate=.08,num_leaves=63,min_child_samples=100,reg_lambda=1.,
            subsample=.85,colsample_bytree=.9,random_state=SEED,n_jobs=4,verbosity=-1)
def model(params=None):
    p=PARAMS.copy()
    if params:p.update(params)
    return LGBMRegressor(**p)

date_stats=test.groupby("time").agg(n=("ID","size"),known=("TWS_t","count"))
KNOWN_POS=set(np.flatnonzero(date_stats.known.to_numpy()>date_stats.n.to_numpy()*.5))
def temporal_splits(df,n_splits=3,block=18):
    dates=np.array(sorted(df.time.unique())); start=len(dates)-n_splits*block
    return [(np.flatnonzero(df.time.isin(dates[:start+i*block])),
             np.flatnonzero(df.time.isin(dates[start+i*block:start+(i+1)*block]))) for i in range(n_splits)]
SPLITS=temporal_splits(train_f)[-2:]
folds=[]
for f,(ti,vi) in enumerate(SPLITS,1):
    tr,va=train_f.iloc[ti].copy(),train_f.iloc[vi].copy()
    vd=np.array(sorted(va.time.unique()))
    known=va.time.isin({d for j,d in enumerate(vd) if j in KNOWN_POS}).to_numpy()
    folds.append({"fold":f,"train":tr,"valid":va,"known":known})
    print(f,tr.time.min(),tr.time.max(),va.time.min(),va.time.max(),known.sum(),(~known).sum())

1 2002-05-01 00:00:00 2011-05-01 00:00:00 2011-08-01 00:00:00 2013-07-01 00:00:00 93511 187425


2 2002-05-01 00:00:00 2013-07-01 00:00:00 2013-11-01 00:00:00 2015-08-01 00:00:00 93638 187424


In [4]:
t=perf_counter()
known_parts=[]; base_masked_parts=[]; base_models=[]
for f in folds:
    km=model().fit(f["train"][BASE_FEATURES],f["train"].target)
    kp=km.predict(f["valid"].loc[f["known"],BASE_FEATURES])
    known_parts.append(pd.DataFrame({"idx":f["valid"].index[f["known"]],"fold":f["fold"],
                                     "target":f["valid"].target.to_numpy()[f["known"]],"pred":kp}))
    mm=model().fit(f["train"][BASE_MASKED],f["train"].target)
    mp=mm.predict(f["valid"].loc[~f["known"],BASE_MASKED])
    base_models.append(mm)
    base_masked_parts.append(pd.DataFrame({"idx":f["valid"].index[~f["known"]],"fold":f["fold"],
        "target":f["valid"].target.to_numpy()[~f["known"]],"pred":mp}))
KNOWN_OOF=pd.concat(known_parts,ignore_index=True); BASE_MOOF=pd.concat(base_masked_parts,ignore_index=True)
def metrics(moof):
    allp=pd.concat([KNOWN_OOF[["idx","fold","target","pred"]],moof[["idx","fold","target","pred"]]],ignore_index=True)
    fr=allp.groupby("fold").apply(lambda z:root_mean_squared_error(z.target,z.pred),include_groups=False)
    return {"Known RMSE":root_mean_squared_error(KNOWN_OOF.target,KNOWN_OOF.pred),
            "Masked RMSE":root_mean_squared_error(moof.target,moof.pred),
            "Overall RMSE":root_mean_squared_error(allp.target,allp.pred),
            "Worst Fold":fr.max(),"Fold Std":fr.std(ddof=0)}
BASE_MET=metrics(BASE_MOOF); display(pd.DataFrame([{"Feature":"Notebook 11 baseline",**BASE_MET}]))
np.savez_compressed(ART/"baseline_oof.npz",idx=BASE_MOOF.idx,pred=BASE_MOOF.pred,target=BASE_MOOF.target,fold=BASE_MOOF.fold)
t=stage("Notebook 11 baseline reproduction",t)

,Feature,Known RMSE,Masked RMSE,Overall RMSE,Worst Fold,Fold Std
0,Notebook 11 baseline,0.608198,0.761184,0.713889,0.775083,0.064092


[STAGE] Notebook 11 baseline reproduction: 29.6s | total=37.9s


## 2. Single-feature interaction and transform ablation

Her satır baseline'a **yalnız tek yeni sütun** ekler. Ratio paydayı güvenli epsilon ile korur.
Log/sqrt işaret-korumalıdır. Yeo–Johnson fold train'de fit edilir. Box–Cox yalnız kesin pozitif
değişkenlerde anlamlıdır; mevcut signed standardized değişkenlerde shift'in test-genellemesi
keyfi olduğundan uygulanmaz.

In [5]:
EPS=1e-4
def add_simple(o,name):
    o=o.copy()
    specs={
      "soil_x_spei01":lambda x:x.SOIL_MOISTURE_t*x.SPEI_01_t,
      "soil_x_spei03":lambda x:x.SOIL_MOISTURE_t*x.SPEI_03_t,
      "soil_x_spei06":lambda x:x.SOIL_MOISTURE_t*x.SPEI_06_t,
      "soil_x_spei12":lambda x:x.SOIL_MOISTURE_t*x.SPEI_12_t,
      "soil_x_spei_mean":lambda x:x.SOIL_MOISTURE_t*x.spei_mean,
      "soil_minus_spei_mean":lambda x:x.SOIL_MOISTURE_t-x.spei_mean,
      "soil_ratio_spei_mean":lambda x:x.SOIL_MOISTURE_t/(x.spei_mean.abs()+EPS),
      "tws_x_spei_mean":lambda x:x.TWS_t*x.spei_mean,
      "lat_x_month":lambda x:x.lat*x.month,
      "lon_x_month":lambda x:x.lon*x.month,
      "month_x_soil":lambda x:x.month*x.SOIL_MOISTURE_t,
      "lat_x_soil":lambda x:x.lat*x.SOIL_MOISTURE_t,
      "lon_x_soil":lambda x:x.lon*x.SOIL_MOISTURE_t,
      "sinmonth_x_spei":lambda x:x.month_sin_calc*x.spei_mean,
      "cosmonth_x_spei":lambda x:x.month_cos_calc*x.spei_mean,
      "soil_abs":lambda x:x.SOIL_MOISTURE_t.abs(),
      "soil_signed_log1p":lambda x:np.sign(x.SOIL_MOISTURE_t)*np.log1p(x.SOIL_MOISTURE_t.abs()),
      "soil_signed_sqrt":lambda x:np.sign(x.SOIL_MOISTURE_t)*np.sqrt(x.SOIL_MOISTURE_t.abs()),
      "spei_abs":lambda x:x.spei_mean.abs(),
      "spei_signed_log1p":lambda x:np.sign(x.spei_mean)*np.log1p(x.spei_mean.abs()),
      "spei_signed_sqrt":lambda x:np.sign(x.spei_mean)*np.sqrt(x.spei_mean.abs()),
    }
    o[name]=specs[name](o).replace([np.inf,-np.inf],np.nan).astype("float32")
    return o

SIMPLE=list({
 "soil_x_spei01":0,"soil_x_spei03":0,"soil_x_spei06":0,"soil_x_spei12":0,
 "soil_x_spei_mean":0,"soil_minus_spei_mean":0,"soil_ratio_spei_mean":0,
 "lat_x_month":0,"lon_x_month":0,"month_x_soil":0,"lat_x_soil":0,"lon_x_soil":0,
 "sinmonth_x_spei":0,"cosmonth_x_spei":0,"soil_abs":0,"soil_signed_log1p":0,
 "soil_signed_sqrt":0,"spei_abs":0,"spei_signed_log1p":0,"spei_signed_sqrt":0})
# TWS interaction masked uzman için kullanılamaz; known uzman sabit tutulduğundan rejected.
NOT_APPLICABLE=[
 {"Feature":"soil_tp1_minus_soil_t","Reason":"SOIL_MOISTURE_tp1 şemada yok; geleceği türetmek leakage"},
 {"Feature":"soil_tp1_ratio_soil_t","Reason":"SOIL_MOISTURE_tp1 şemada yok; geleceği türetmek leakage"},
 {"Feature":"soil_tp1_x_spei","Reason":"SOIL_MOISTURE_tp1 şemada yok; geleceği türetmek leakage"},
 {"Feature":"tws_x_spei_mean","Reason":"Masked test segmentinde TWS_t NaN; masked uzman için kullanılamaz"},
 {"Feature":"boxcox","Reason":"Aday değişkenler signed; foldlar arası keyfi pozitif shift güvenli değil"}]
display(pd.DataFrame(NOT_APPLICABLE))

,Feature,Reason
0,soil_tp1_minus_soil_t,SOIL_MOISTURE_tp1 şemada yok; geleceği türetme...
1,soil_tp1_ratio_soil_t,SOIL_MOISTURE_tp1 şemada yok; geleceği türetme...
2,soil_tp1_x_spei,SOIL_MOISTURE_tp1 şemada yok; geleceği türetme...
3,tws_x_spei_mean,Masked test segmentinde TWS_t NaN; masked uzma...
4,boxcox,Aday değişkenler signed; foldlar arası keyfi p...


In [6]:
def eval_candidate(name,builder,extra_cols,params=None):
    start=perf_counter()
    cache_path=ART/f"oof_{name.replace('/','_')}.pkl"
    if cache_path.exists():
        moof=pd.read_pickle(cache_path); met=metrics(moof)
        row={"Feature":name,**met,"OOF Gain":BASE_MET["Overall RMSE"]-met["Overall RMSE"],
             "Accepted":met["Overall RMSE"]<BASE_MET["Overall RMSE"] and met["Worst Fold"]<=BASE_MET["Worst Fold"]+.002,
             "Training sec":0.0}
        print(name,row["Overall RMSE"],row["OOF Gain"],row["Accepted"],"[checkpoint]")
        return row
    parts=[]
    for f in folds:
        tr,va=builder(f["train"],f["valid"])
        feats=BASE_MASKED+extra_cols
        m=model(params).fit(tr[feats],tr.target)
        pred=m.predict(va.loc[~f["known"],feats])
        parts.append(pd.DataFrame({"idx":f["valid"].index[~f["known"]],"fold":f["fold"],
          "target":f["valid"].target.to_numpy()[~f["known"]],"pred":pred}))
        del m,tr,va; gc.collect()
    moof=pd.concat(parts,ignore_index=True); met=metrics(moof)
    row={"Feature":name,**met,"OOF Gain":BASE_MET["Overall RMSE"]-met["Overall RMSE"],
         "Accepted":met["Overall RMSE"]<BASE_MET["Overall RMSE"] and met["Worst Fold"]<=BASE_MET["Worst Fold"]+.002,
         "Training sec":perf_counter()-start}
    moof.to_pickle(cache_path)
    print(name,row["Overall RMSE"],row["OOF Gain"],row["Accepted"])
    return row

t=perf_counter(); RESULTS=[]
for name in SIMPLE:
    RESULTS.append(eval_candidate(name,lambda tr,va,n=name:(add_simple(tr,n),add_simple(va,n)),[name]))

def yj_builder(tr,va):
    tr=tr.copy(); va=va.copy(); pt=PowerTransformer(method="yeo-johnson",standardize=True)
    tr["soil_yeojohnson"]=pt.fit_transform(tr[["SOIL_MOISTURE_t"]]).ravel().astype("float32")
    va["soil_yeojohnson"]=pt.transform(va[["SOIL_MOISTURE_t"]]).ravel().astype("float32")
    return tr,va
RESULTS.append(eval_candidate("soil_yeojohnson",yj_builder,["soil_yeojohnson"]))
pd.DataFrame(RESULTS).to_csv(ART/"ablation_interactions.csv",index=False)
display(pd.DataFrame(RESULTS).sort_values("Overall RMSE"))
t=stage("single interaction/transform ablation",t)

soil_x_spei01 0.7142562900665029 -0.0003676494384871054 False [checkpoint]
soil_x_spei03 0.714713189838961 -0.0008245492109452135 False [checkpoint]
soil_x_spei06 0.7159716499570463 -0.002083009329030494 False [checkpoint]


soil_x_spei12 0.7153013880918275 -0.0014127474638117032 False [checkpoint]
soil_x_spei_mean 0.7160274964572608 -0.002138855829245001 False [checkpoint]


soil_minus_spei_mean 0.7156712769758242 -0.0017826363478083973 False [checkpoint]
soil_ratio_spei_mean 0.715605561301801 -0.0017169206737851495 False [checkpoint]
lat_x_month 0.7169977274706568 -0.0031090868426409957 False [checkpoint]


lon_x_month 0.7182962830227119 -0.004407642394696043 False [checkpoint]
month_x_soil 0.7153948879112685 -0.0015062472832526597 False [checkpoint]


lat_x_soil 0.7135751378351601 0.00031350279285569105 True [checkpoint]
lon_x_soil 0.7172163311220241 -0.003327690494008273 False [checkpoint]
sinmonth_x_spei 0.7141147421707279 -0.00022610154271207605 False [checkpoint]


cosmonth_x_spei 0.7144170530361962 -0.0005284124081803565 False [checkpoint]
soil_abs 0.7154000536603375 -0.0015114130323217179 False [checkpoint]


soil_signed_log1p 0.7183234698255863 -0.0044348291975704335 False [checkpoint]
soil_signed_sqrt 0.7183234698255863 -0.0044348291975704335 False [checkpoint]
spei_abs 0.7183426270712565 -0.004453986443240687 False [checkpoint]


spei_signed_log1p 0.7156494560619783 -0.0017608154339624482 False [checkpoint]
spei_signed_sqrt 0.7156494560619783 -0.0017608154339624482 False [checkpoint]


soil_yeojohnson 0.7181525032251105 -0.004263862597094659 False [checkpoint]


,Feature,Known RMSE,Masked RMSE,Overall RMSE,Worst Fold,Fold Std,OOF Gain,Accepted,Training sec
10,lat_x_soil,0.608198,0.760743,0.713575,0.774703,0.064019,0.000314,True,0.0
12,sinmonth_x_spei,0.608198,0.761501,0.714115,0.775948,0.064794,-0.000226,False,0.0
0,soil_x_spei01,0.608198,0.761700,0.714256,0.775039,0.063637,-0.000368,False,0.0
13,cosmonth_x_spei,0.608198,0.761926,0.714417,0.775563,0.064036,-0.000528,False,0.0
1,soil_x_spei03,0.608198,0.762343,0.714713,0.778805,0.067281,-0.000825,False,0.0
3,soil_x_spei12,0.608198,0.763169,0.715301,0.776064,0.063611,-0.001413,False,0.0
9,month_x_soil,0.608198,0.763301,0.715395,0.777488,0.065073,-0.001506,False,0.0
14,soil_abs,0.608198,0.763308,0.715400,0.777327,0.064891,-0.001511,False,0.0
6,soil_ratio_spei_mean,0.608198,0.763597,0.715606,0.777953,0.065352,-0.001717,False,0.0
19,spei_signed_sqrt,0.608198,0.763658,0.715649,0.777365,0.064657,-0.001761,False,0.0


[STAGE] single interaction/transform ablation: 1.0s | total=39.0s


## 3. Top-10 polynomial interaction-only experiment

In [7]:
t=perf_counter()
imp=pd.DataFrame({"feature":BASE_MASKED,"gain":np.mean([m.booster_.feature_importance(importance_type="gain") for m in base_models],axis=0)})
top10=imp.sort_values("gain",ascending=False).head(10).feature.tolist()
poly=PolynomialFeatures(degree=2,interaction_only=True,include_bias=False)
poly.fit(train_f[top10].iloc[:100])
pair_names=[x.replace(" ","__x__") for x in poly.get_feature_names_out(top10) if " " in x]
def poly_builder(tr,va):
    tr=tr.copy(); va=va.copy()
    for nm in pair_names:
        a,b=nm.split("__x__"); tr[nm]=(tr[a]*tr[b]).astype("float32"); va[nm]=(va[a]*va[b]).astype("float32")
    return tr,va
poly_row=eval_candidate("poly_top10_degree2_interaction_only",poly_builder,pair_names)
RESULTS.append(poly_row)
print("top10",top10,"pair count",len(pair_names))
t=stage("polynomial",t)

poly_top10_degree2_interaction_only 0.7220877859062298 -0.00819914527821397 False [checkpoint]
top10 ['elapsed_month', 'spatial_bin_10', 'lat', 'SPEI_12_t', 'lat_lon_interaction', 'lon', 'spei_mean', 'lon_cos', 'SPEI_06_t', 'lon_sin'] pair count 45
[STAGE] polynomial: 0.1s | total=39.1s


## 4. Spatial clusters, bands, density and leakage-safe frequency/target encoding

In [8]:
def spatial_builder(kind,k):
    def build(tr,va):
        tr=tr.copy(); va=va.copy()
        cls=KMeans(n_clusters=k,random_state=SEED,n_init=10) if kind=="kmeans" else MiniBatchKMeans(n_clusters=k,random_state=SEED,batch_size=4096,n_init=5)
        # unique coordinates prevent repeated months from changing spatial weighting
        coords=tr[["lat","lon"]].drop_duplicates().to_numpy()
        cls.fit(coords)
        for o in [tr,va]:
            lab=cls.predict(o[["lat","lon"]]); o[f"{kind}_{k}_id"]=lab.astype("int16")
            o[f"{kind}_{k}_dist"]=np.linalg.norm(o[["lat","lon"]].to_numpy()-cls.cluster_centers_[lab],axis=1).astype("float32")
        counts=pd.Series(cls.labels_).value_counts()
        tr[f"{kind}_{k}_density"]=tr[f"{kind}_{k}_id"].map(counts).astype("float32")
        va[f"{kind}_{k}_density"]=va[f"{kind}_{k}_id"].map(counts).fillna(0).astype("float32")
        return tr,va
    return build

t=perf_counter()
for kind in ["kmeans","mbkmeans"]:
    for k in [8,16,32,64]:
        b=spatial_builder(kind,k)
        for suffix in ["id","dist","density"]:
            name=f"{kind}_{k}_{suffix}"; RESULTS.append(eval_candidate(name,b,[name]))

def band_builder(tr,va):
    tr=tr.copy(); va=va.copy()
    for o in [tr,va]:
        o["lat_band_2"]=np.floor((o.lat+90)/2).astype("int16")
        o["lon_band_2"]=np.floor((o.lon+180)/2).astype("int16")
        o["quadrant"]=((o.lat>=0).astype("int8")*2+(o.lon>=0).astype("int8"))
    return tr,va
for name in ["lat_band_2","lon_band_2","quadrant"]:
    RESULTS.append(eval_candidate(name,band_builder,[name]))
t=stage("spatial ablations",t)

kmeans_8_id 0.7159385678704037 -0.002049927242387839 False [checkpoint]


kmeans_8_dist 0.7203388851506737 -0.006450244522657855 False [checkpoint]


kmeans_8_density 0.7148856326957834 -0.0009969920677675992 False [checkpoint]


kmeans_16_id 0.7146672726931924 -0.0007786320651765877 False [checkpoint]
kmeans_16_dist 0.7168550142073649 -0.0029663735793490353 False [checkpoint]


kmeans_16_density 0.7174238570690425 -0.003535216441026723 False [checkpoint]


kmeans_32_id 0.7174808844459595 -0.0035922438179436522 False [checkpoint]


... (12 outputs left out)

mbkmeans_64_dist 0.7153516660113706 -0.0014630253833547435 False [checkpoint]


mbkmeans_64_density 0.719370683196037 -0.005482042568021206 False [checkpoint]
lat_band_2 0.7156050619813772 -0.0017164213533613637 False [checkpoint]


lon_band_2 0.7142701281583367 -0.00038148753032085647 False [checkpoint]


quadrant 0.7167430214975445 -0.0028543808695287165 False [checkpoint]
[STAGE] spatial ablations: 1.3s | total=40.4s


In [9]:
t=perf_counter()
def encoding_builder(group,name,target_encode=False,cluster=None):
    def build(tr,va):
        tr=tr.copy(); va=va.copy()
        g=group
        if cluster:
            tr,va=spatial_builder("mbkmeans",cluster)(tr,va); g=[f"mbkmeans_{cluster}_id"]
        keys=g if isinstance(g,list) else [g]
        if target_encode:
            global_mean=tr.target.mean(); stat=tr.groupby(keys).target.agg(["mean","count"])
            smooth=(stat["mean"]*stat["count"]+global_mean*20)/(stat["count"]+20)
            # leave-one-out train encoding; validation mapping only fold-train
            sums=tr.groupby(keys).target.transform("sum"); cnt=tr.groupby(keys).target.transform("count")
            tr[name]=((sums-tr.target+20*global_mean)/(cnt-1+20)).astype("float32")
            va[name]=pd.MultiIndex.from_frame(va[keys]).map(smooth).fillna(global_mean).astype("float32")
        else:
            freq=tr.groupby(keys).size()/len(tr)
            tr[name]=pd.MultiIndex.from_frame(tr[keys]).map(freq).astype("float32")
            va[name]=pd.MultiIndex.from_frame(va[keys]).map(freq).fillna(0).astype("float32")
        return tr,va
    return build

freq_specs=[("location_frequency",["lat","lon"],None),("month_frequency",["month"],None),
 ("lat_band_frequency",["lat_band_5"],None),("lon_band_frequency",["lon_band_5"],None),
 ("cluster_frequency",None,32)]
for name,g,k in freq_specs:
    RESULTS.append(eval_candidate(name,encoding_builder(g,name,False,k),[name]))

te_specs=[("te_month",["month"],None),("te_cluster",None,32),("te_lat_band",["lat_band_5"],None),
          ("te_lon_band",["lon_band_5"],None),("te_region",["lat_band_5","lon_band_5"],None)]
for name,g,k in te_specs:
    RESULTS.append(eval_candidate(name,encoding_builder(g,name,True,k),[name]))
t=stage("frequency + OOF target encoding",t)

location_frequency 0.7165907584670551 -0.0027021178390392864 False [checkpoint]


month_frequency 0.7138886406280158 0.0 False [checkpoint]
lat_band_frequency 0.7138886406280158 0.0 False [checkpoint]


lon_band_frequency 0.7138886406280158 0.0 False [checkpoint]


cluster_frequency 0.7138886406280158 0.0 False [checkpoint]


te_month 1.207125479046677 -0.49323683841866117 False [checkpoint]


te_cluster 1.0204095022864617 -0.30652086165844583 False [checkpoint]
te_lat_band 1.185725014517159 -0.4718363738891431 False [checkpoint]


te_lon_band 0.8243824425381802 -0.11049380191016434 False [checkpoint]


te_region 0.7171732050866968 -0.003284564458680972 False [checkpoint]
[STAGE] frequency + OOF target encoding: 0.5s | total=40.9s


## 5. Feature selection, monotonic constraints, uncertainty and residual research

In [10]:
t=perf_counter()
ablation=pd.DataFrame(RESULTS).sort_values("Overall RMSE").reset_index(drop=True)
ablation.to_csv(ART/"ablation_all.csv",index=False)
accepted=ablation.loc[ablation.Accepted,"Feature"].tolist()
display(ablation)

# Built-in importance + permutation on a bounded validation sample.
f=folds[-1]; bm=base_models[-1]; rng=np.random.default_rng(SEED)
ix=rng.choice(len(f["valid"]),min(10000,len(f["valid"])),replace=False)
pi=permutation_importance(bm,f["valid"].iloc[ix][BASE_MASKED],f["valid"].iloc[ix].target,
                          scoring="neg_root_mean_squared_error",n_repeats=3,random_state=SEED,n_jobs=1)
selection=pd.DataFrame({"Feature":BASE_MASKED,
 "LGBM gain":bm.booster_.feature_importance(importance_type="gain"),
 "Permutation importance":pi.importances_mean}).sort_values("LGBM gain",ascending=False)
try:
    import shap
    sx=f["valid"].iloc[ix[:2000]][BASE_MASKED]
    sv=shap.TreeExplainer(bm).shap_values(sx)
    shap_map=dict(zip(BASE_MASKED,np.abs(sv).mean(0)))
    selection["Mean abs SHAP"]=selection.Feature.map(shap_map)
except Exception as e:
    selection["Mean abs SHAP"]=np.nan; print("SHAP unavailable:",repr(e))
display(selection)

drop_candidates=selection.tail(5).Feature.tolist()
def drop_builder(tr,va): return tr.copy(),va.copy()
drop_row=eval_candidate("prune_bottom5_importance",drop_builder,[])
# special evaluation with removed features
def eval_pruned():
    global BASE_MASKED
    old=BASE_MASKED; BASE_MASKED=[x for x in old if x not in drop_candidates]
    try:return eval_candidate("prune_bottom5",drop_builder,[])
    finally:BASE_MASKED=old
prune_row=eval_pruned(); RESULTS.append(prune_row)

# Domain-defensible constraints only: SPEI and soil moisture signs are tested, not assumed selected.
constraints=[1 if x in SPEI+["SOIL_MOISTURE_t","spei_mean","spei_min","spei_max"] else 0 for x in BASE_MASKED]
mono_row=eval_candidate("monotonic_domain_signs",drop_builder,[],{"monotone_constraints":constraints})
RESULTS.append(mono_row)
t=stage("importance + selection + monotonic",t)

,Feature,Known RMSE,Masked RMSE,Overall RMSE,Worst Fold,Fold Std,OOF Gain,Accepted,Training sec
0,mbkmeans_8_dist,0.608198,0.757455,0.711239,0.767996,0.059243,0.002650,True,0.0
1,lat_x_soil,0.608198,0.760743,0.713575,0.774703,0.064019,0.000314,True,0.0
2,kmeans_32_density,0.608198,0.761032,0.713781,0.776061,0.065287,0.000108,True,0.0
3,cluster_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
4,lat_band_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
5,month_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
6,lon_band_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
7,mbkmeans_16_density,0.608198,0.761317,0.713983,0.775448,0.064388,-0.000095,False,0.0
8,kmeans_32_dist,0.608198,0.761477,0.714097,0.774378,0.063087,-0.000209,False,0.0
9,mbkmeans_32_density,0.608198,0.761500,0.714113,0.776284,0.065165,-0.000225,False,0.0


,Feature,LGBM gain,Permutation importance,Mean abs SHAP
12,elapsed_month,1.048101e+06,0.000000,0.176691
26,spatial_bin_10,9.701528e+05,-0.008773,0.073034
5,SPEI_12_t,7.101654e+05,0.032389,0.110410
0,lat,6.312533e+05,-0.009867,0.050262
27,lat_lon_interaction,5.961446e+05,-0.017673,0.050081
1,lon,5.129081e+05,-0.020532,0.040767
17,spei_mean,4.740633e+05,0.013149,0.065364
31,lon_cos,3.843352e+05,-0.008859,0.037698
4,SPEI_06_t,3.683095e+05,0.020088,0.085193
30,lon_sin,3.296234e+05,-0.008791,0.029065


prune_bottom5_importance 0.7138886406280158 0.0 False [checkpoint]
prune_bottom5 0.7153077418681465 -0.0014191012401306713 False [checkpoint]
monotonic_domain_signs 0.7167361630515247 -0.0028475224235088303 False [checkpoint]
[STAGE] importance + selection + monotonic: 4.1s | total=45.0s


In [11]:
t=perf_counter()
# Baseline residual slices.
res=BASE_MOOF.merge(train_f[["month","lat","lon","spei_mean","SOIL_MOISTURE_t"]],left_on="idx",right_index=True)
res["abs_error"]=(res.target-res.pred).abs()
res["lat_band"]=pd.cut(res.lat,8); res["spei_bin"]=pd.qcut(res.spei_mean,8,duplicates="drop")
res["soil_bin"]=pd.qcut(res.SOIL_MOISTURE_t,8,duplicates="drop")
residual_tables={}
for col in ["month","lat_band","spei_bin","soil_bin"]:
    residual_tables[col]=res.groupby(col,observed=True).agg(n=("target","size"),
      RMSE=("abs_error",lambda x:np.sqrt(np.mean(x*x))),MAE=("abs_error","mean")).sort_values("RMSE",ascending=False)
    display(Markdown(f"**Residual by {col}**")); display(residual_tables[col].head(10))

# Fold-model disagreement is a genuine uncertainty proxy.
sample_x=train_f.loc[BASE_MOOF.idx,BASE_MASKED]
P=np.column_stack([m.predict(sample_x) for m in base_models])
unc=P.std(axis=1); err=np.abs(BASE_MOOF.target.to_numpy()-BASE_MOOF.pred.to_numpy())
q=pd.qcut(unc,5,labels=False,duplicates="drop")
uncertainty=pd.DataFrame({"uncertainty":unc,"abs_error":err,"quintile":q}).groupby("quintile").agg(
 n=("abs_error","size"),mean_uncertainty=("uncertainty","mean"),MAE=("abs_error","mean"))
display(uncertainty)

# Pseudo labeling decision: no true label exists for test; confidence alone cannot establish benefit.
# Simulated self-training adds no independent information and compounds error, so it is deliberately not applied.
pseudo_decision={"applied":False,"reason":"OOF ile doğrulanabilir bağımsız pseudo-label kaynağı yok; self-training riskli"}
print(pseudo_decision)
t=stage("residual + uncertainty + pseudo-label decision",t)

**Residual by month**

,n,RMSE,MAE
month,,,
2,47001,0.880015,0.660420
5,15583,0.823016,0.666770
12,31264,0.804129,0.630696
1,62597,0.781272,0.609721
8,31067,0.779647,0.611254
11,15648,0.753607,0.584692
4,31289,0.728800,0.574893
6,15579,0.715078,0.543634
7,31166,0.708481,0.558246


**Residual by lat_band**

,n,RMSE,MAE
lat_band,,,
"(-38.125, -20.75]",26987,0.824767,0.648911
"(-20.75, -3.375]",35851,0.822363,0.640578
"(14.0, 31.375]",49819,0.806908,0.622531
"(-55.639, -38.125]",4578,0.792165,0.591911
"(48.75, 66.125]",92922,0.763050,0.604495
"(-3.375, 14.0]",34544,0.747783,0.581093
"(31.375, 48.75]",74289,0.725469,0.561454
"(66.125, 83.5]",55859,0.692096,0.530848


**Residual by spei_bin**

,n,RMSE,MAE
spei_bin,,,
"(0.828, 2.66]",46856,0.787865,0.620366
"(0.457, 0.828]",46856,0.771200,0.604047
"(-2.631, -0.982]",46857,0.769166,0.602773
"(0.168, 0.457]",46856,0.761338,0.592827
"(-0.982, -0.628]",46856,0.760825,0.585776
"(-0.628, -0.348]",46856,0.751009,0.579116
"(-0.0909, 0.168]",46856,0.747408,0.578405
"(-0.348, -0.0909]",46856,0.739585,0.570494


**Residual by soil_bin**

,n,RMSE,MAE
soil_bin,,,
"(-0.451, -0.23]",46856,0.784310,0.596051
"(0.79, 5.385]",46856,0.768395,0.609320
"(-4.713, -0.834]",46857,0.765520,0.599784
"(-0.00138, 0.248]",46856,0.761229,0.590859
"(-0.834, -0.451]",46856,0.755167,0.584848
"(-0.23, -0.00138]",46856,0.754960,0.581521
"(0.248, 0.49]",46856,0.751644,0.586605
"(0.49, 0.79]",46856,0.747615,0.584817


,n,mean_uncertainty,MAE
quintile,,,
0,74970,0.025141,0.511254
1,74970,0.076702,0.527116
2,74969,0.131439,0.566659
3,74970,0.199375,0.629317
4,74970,0.350850,0.724282


{'applied': False, 'reason': 'OOF ile doğrulanabilir bağımsız pseudo-label kaynağı yok; self-training riskli'}
[STAGE] residual + uncertainty + pseudo-label decision: 1.8s | total=46.8s


## 6. Accepted features together, conservative blend, full fit and submissions

In [12]:
# Rebuild accepted candidates by name. To control multiple-testing risk, require positive gain and
# no material worst-fold regression; then test the combined representation once.
accepted=ablation.query("Accepted").Feature.tolist()
simple_acc=[x for x in accepted if x in SIMPLE]
print("Accepted single features",accepted)

def combined_builder(tr,va):
    tr=tr.copy(); va=va.copy()
    extras=[]
    for n in simple_acc: tr=add_simple(tr,n); va=add_simple(va,n); extras.append(n)
    return tr,va

combined_row=eval_candidate("accepted_simple_combined",combined_builder,simple_acc) if simple_acc else None
if combined_row: RESULTS.append(combined_row)
final_table=pd.DataFrame(RESULTS).sort_values("Overall RMSE")
best_row=final_table.iloc[0]
if best_row["Overall RMSE"]>=BASE_MET["Overall RMSE"]: best_name="Notebook 11 baseline"
else: best_name=best_row.Feature
print("Final selection",best_name); display(final_table.head(20))

def load_oof(name):
    if name=="Notebook 11 baseline": return BASE_MOOF.copy()
    return pd.read_pickle(ART/f"oof_{name.replace('/','_')}.pkl")
best_oof=load_oof(best_name)
align=BASE_MOOF[["idx","fold","target","pred"]].merge(best_oof[["idx","pred"]],on="idx",suffixes=("_base","_new"))
blend_grid=[]
for w in np.arange(0,1.001,.05):
    p=w*align.pred_new+(1-w)*align.pred_base
    z=align[["idx","fold","target"]].copy(); z["pred"]=p
    blend_grid.append({"new_weight":w,**metrics(z)})
blend_grid=pd.DataFrame(blend_grid).sort_values("Overall RMSE")
display(blend_grid.head())

Accepted single features ['mbkmeans_8_dist', 'lat_x_soil', 'kmeans_32_density']
accepted_simple_combined 0.7135751378351601 0.00031350279285569105 True [checkpoint]
Final selection mbkmeans_8_dist


,Feature,Known RMSE,Masked RMSE,Overall RMSE,Worst Fold,Fold Std,OOF Gain,Accepted,Training sec
35,mbkmeans_8_dist,0.608198,0.757455,0.711239,0.767996,0.059243,0.002650,True,0.0
10,lat_x_soil,0.608198,0.760743,0.713575,0.774703,0.064019,0.000314,True,0.0
61,accepted_simple_combined,0.608198,0.760743,0.713575,0.774703,0.064019,0.000314,True,0.0
30,kmeans_32_density,0.608198,0.761032,0.713781,0.776061,0.065287,0.000108,True,0.0
53,cluster_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
51,lat_band_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
52,lon_band_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
50,month_frequency,0.608198,0.761184,0.713889,0.775083,0.064092,0.000000,False,0.0
39,mbkmeans_16_density,0.608198,0.761317,0.713983,0.775448,0.064388,-0.000095,False,0.0
29,kmeans_32_dist,0.608198,0.761477,0.714097,0.774378,0.063087,-0.000209,False,0.0


,new_weight,Known RMSE,Masked RMSE,Overall RMSE,Worst Fold,Fold Std
14,0.70,0.608198,0.756421,0.710504,0.768553,0.060656
13,0.65,0.608198,0.756429,0.710510,0.768777,0.060896
15,0.75,0.608198,0.756465,0.710536,0.768366,0.060418
12,0.60,0.608198,0.756487,0.710551,0.769039,0.061136
16,0.80,0.608198,0.756561,0.710603,0.768217,0.060180


In [13]:
t=perf_counter()
known_test=test_f.TWS_t.notna().to_numpy()
known_model=model().fit(train_f[BASE_FEATURES],train_f.target)
known_pred=known_model.predict(test_f.loc[known_test,BASE_FEATURES])
base_full=model().fit(train_f[BASE_MASKED],train_f.target)
base_test=base_full.predict(test_f.loc[~known_test,BASE_MASKED])

def fit_selected(name):
    if name=="Notebook 11 baseline": return base_test
    if name in SIMPLE:
        tr=add_simple(train_f,name); te=add_simple(test_f,name); feats=BASE_MASKED+[name]
    elif name=="accepted_simple_combined":
        tr,te=combined_builder(train_f,test_f); feats=BASE_MASKED+simple_acc
    elif name.startswith(("kmeans_","mbkmeans_")):
        bits=name.split("_"); kind=bits[0]; k=int(bits[1])
        tr,te=spatial_builder(kind,k)(train_f,test_f); feats=BASE_MASKED+[name]
    else:
        # Complex fold-fit candidates are retained as research evidence; conservative final fallback
        # prevents a different full-data preprocessing path from silently diverging.
        print("Complex winner uses Notebook 11 final fallback:",name); return base_test
    return model().fit(tr[feats],tr.target).predict(te.loc[~known_test,feats])

new_test=fit_selected(best_name)
w=float(blend_grid.iloc[0].new_weight); blend_test=w*new_test+(1-w)*base_test
def assemble(masked):
    p=np.empty(len(test_f)); p[known_test]=known_pred; p[~known_test]=masked; return p
candidates={"A_best_single":assemble(new_test),"B_conservative_blend":assemble(blend_test),
            "C_notebook11_diverse":assemble(base_test)}

def validate_write(label,pred):
    pred=np.asarray(pred,float); byid=pd.Series(pred,index=test_f.ID.astype(str))
    out=sample.copy(); out["Target"]=sample.ID.astype(str).map(byid).to_numpy(float)
    checks={"rows_280961":len(out)==280961,"id_order":out.ID.equals(sample.ID),
      "duplicate":not out.ID.duplicated().any(),"nan":not out.Target.isna().any(),
      "infinity":np.isfinite(out.Target).all(),"numeric":pd.api.types.is_numeric_dtype(out.Target)}
    assert all(checks.values())
    path=SUB/f"submission_16_{label}.csv"; out.to_csv(path,index=False)
    return out,path,{**checks,"min":out.Target.min(),"max":out.Target.max(),"mean":out.Target.mean(),"std":out.Target.std()}

predcorr=pd.DataFrame(candidates).corr(); display(predcorr)
# En fazla üç; %99.5'ten yüksek korelasyonlu kopyaları yazma.
written={}; reports=[]; kept=[]
for label,pred in candidates.items():
    if any(abs(np.corrcoef(pred,candidates[k])[0,1])>=.995 for k in kept):
        print("Skipped near-duplicate",label); continue
    out,path,rep=validate_write(label,pred); kept.append(label); written[label]=(out,path); reports.append({"Submission":label,"path":str(path),**rep})
submission_report=pd.DataFrame(reports); submission_report.to_csv(ART/"submission_report.csv",index=False)
display(submission_report)
t=stage("full fit + submission validation",t)

,A_best_single,B_conservative_blend,C_notebook11_diverse
A_best_single,1.000000,0.998294,0.981025
B_conservative_blend,0.998294,1.000000,0.990671
C_notebook11_diverse,0.981025,0.990671,1.000000


Skipped near-duplicate B_conservative_blend


,Submission,path,rows_280961,id_order,duplicate,nan,infinity,numeric,min,max,mean,std
0,A_best_single,C:\global-water-storage-forecasting\submission...,True,True,True,True,True,True,-2.255716,2.054586,-0.113644,0.540062
1,C_notebook11_diverse,C:\global-water-storage-forecasting\submission...,True,True,True,True,True,True,-2.255716,2.054586,-0.112960,0.539841


[STAGE] full fit + submission validation: 30.9s | total=78.8s


## 7. Final evidence-based report

In [14]:
final_table=pd.DataFrame(RESULTS).sort_values("Overall RMSE")
best=final_table.iloc[0]; worst=final_table.iloc[-1]
sp=final_table[final_table.Feature.str.contains("kmeans|band|quadrant",case=False,regex=True)]
cl=final_table[final_table.Feature.str.contains("kmeans",case=False,regex=True)]
te=final_table[final_table.Feature.str.startswith("te_")]
inter=final_table[final_table.Feature.isin(SIMPLE)]
answers=f'''
### Final rapor

1. **En faydalı yeni feature:** `{best.Feature}`; OOF gain `{best['OOF Gain']:+.6f}`.
2. **En başarısız feature:** `{worst.Feature}`; OOF gain `{worst['OOF Gain']:+.6f}`.
3. **Spatial feature işe yaradı mı?** En iyi spatial gain `{sp['OOF Gain'].max():+.6f}`; karar OOF tablosundaki Accepted alanıdır.
4. **Cluster işe yaradı mı?** En iyi cluster gain `{cl['OOF Gain'].max():+.6f}`.
5. **Target encoding işe yaradı mı?** En iyi leakage-safe TE gain `{te['OOF Gain'].max():+.6f}`.
6. **Interaction katkısı:** En iyi tekil interaction gain `{inter['OOF Gain'].max():+.6f}`; toplu sonuç ayrıca raporlandı.
7. **Pseudo labeling:** Hayır. Gerçek test label'ı veya bağımsız öğretmen yok; confidence self-training'i güvenilir kılmaz.
8. **Monotonic constraints:** Gain `{mono_row['OOF Gain']:+.6f}`; yalnız pozitifse ve worst fold bozulmadıysa tutulur.
9. **Notebook 11 neden güçlü?** Known/masked uzman ayrımı test missingness desenini taklit ediyor; temporal split drift'i
ölçüyor ve düşük varyanslı, fiziksel olarak anlamlı static feature set kullanıyor.
10. **Leaderboard beklentisi:** Yalnız OOF ve worst-fold birlikte iyileşen feature seçildi; yakın-kopya submissionlar
elenerek sahte çeşitlilik önlendi. Buna rağmen public iyileşme garanti değildir; test horizon drift'i bilinmiyor.

Baseline: known `{BASE_MET['Known RMSE']:.6f}`, masked `{BASE_MET['Masked RMSE']:.6f}`,
overall `{BASE_MET['Overall RMSE']:.6f}`, worst `{BASE_MET['Worst Fold']:.6f}`, std `{BASE_MET['Fold Std']:.6f}`.
Toplam süre: `{perf_counter()-T0:.1f}` saniye. Checkpoint: `{ART}`.
'''
display(Markdown(answers))
assert Path(ART/"ablation_all.csv").exists() and len(submission_report)>=1


### Final rapor

1. **En faydalı yeni feature:** `mbkmeans_8_dist`; OOF gain `+0.002650`.
2. **En başarısız feature:** `te_month`; OOF gain `-0.493237`.
3. **Spatial feature işe yaradı mı?** En iyi spatial gain `+0.002650`; karar OOF tablosundaki Accepted alanıdır.
4. **Cluster işe yaradı mı?** En iyi cluster gain `+0.002650`.
5. **Target encoding işe yaradı mı?** En iyi leakage-safe TE gain `-0.003285`.
6. **Interaction katkısı:** En iyi tekil interaction gain `+0.000314`; toplu sonuç ayrıca raporlandı.
7. **Pseudo labeling:** Hayır. Gerçek test label'ı veya bağımsız öğretmen yok; confidence self-training'i güvenilir kılmaz.
8. **Monotonic constraints:** Gain `-0.002848`; yalnız pozitifse ve worst fold bozulmadıysa tutulur.
9. **Notebook 11 neden güçlü?** Known/masked uzman ayrımı test missingness desenini taklit ediyor; temporal split drift'i
ölçüyor ve düşük varyanslı, fiziksel olarak anlamlı static feature set kullanıyor.
10. **Leaderboard beklentisi:** Yalnız OOF ve worst-fold birlikte iyileşen feature seçildi; yakın-kopya submissionlar
elenerek sahte çeşitlilik önlendi. Buna rağmen public iyileşme garanti değildir; test horizon drift'i bilinmiyor.

Baseline: known `0.608198`, masked `0.761184`,
overall `0.713889`, worst `0.775083`, std `0.064092`.
Toplam süre: `78.8` saniye. Checkpoint: `C:\global-water-storage-forecasting\artifacts\notebook16`.
